# 🇳🇬 Naija Persona Agent — Corpus Build via OpenAI (Alternative to 01)

**Same output as `01_build_corpus.ipynb` — different backend.** Use this notebook if NVIDIA NIM rate limits are blocking you and you have OpenAI API budget.

**Run on CPU runtime.** No GPU credits needed.

## Why OpenAI?
- **Much higher rate limits** — tier 1 OpenAI handles 20+ concurrent easily; tier 2+ handles 100+. Free NVIDIA NIM hits 429s at concurrency 6.
- **GPT-4o is excellent at Nigerian Pidgin** — trained on substantially more Nigerian-language web data than European-focused models.
- **Fast end-to-end** — 3k corpus takes ~10-15 min on OpenAI vs 30-40 min on rate-limited NIM.
- **Predictable cost** — see table below.

## Cost estimates (USD)

| Corpus size | gpt-4o-mini (fast/cheap) | gpt-4o (best Pidgin quality) |
|---|---|---|
| 3,000 records | ~$5 | ~$25 |
| 10,000 records | ~$15 | ~$80 |
| 20,000 records | ~$30 | ~$160 |

**For hackathon submission**: use gpt-4o for the headline Pidgin quality. Cost is rounding error vs the value of a defensible cultural-fidelity claim.

## Inputs
- `OPENAI_API_KEY` — set in Colab Secrets

## Outputs (separate Drive root from 01_build_corpus.ipynb)

Stored at `/content/drive/MyDrive/naija_persona_corpus_openai/` so you can keep both
corpora side-by-side and compare. To train on this corpus, set `CORPUS_SOURCE="openai"`
in 02_finetune.ipynb (default).

- `processed/v1_train_alpaca.jsonl`
- `processed/v1_val_alpaca.jsonl`
- `processed/v1_test_alpaca.jsonl`
- `processed/final_alpaca_format.jsonl` + ShareGPT + Parquet
- `augmented/checkpoints/pipeline{1,2,3}/batch_*.parquet`
- `metadata/final_statistics.json`

## 0. Setup & Install

In [ ]:
# Minimal install — no data_designer, no Unsloth conflicts
!pip install -q openai==1.55.* jsonlines pandas tqdm nest_asyncio httpx "pyarrow>=19.0.1,<20" datasets
print("✅ Dependencies installed.")

In [ ]:
# IMPORTS
import os, sys, json, time, csv, asyncio, traceback, random, re
from pathlib import Path
from datetime import datetime
from collections import Counter, defaultdict
import pandas as pd
import jsonlines
import httpx
import nest_asyncio; nest_asyncio.apply()
from tqdm.notebook import tqdm

from openai import AsyncOpenAI

IN_COLAB = "google.colab" in sys.modules
print("✅ Imports loaded.")

In [ ]:
# MOUNT GOOGLE DRIVE — same path as 01_build_corpus.ipynb
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    DRIVE_BASE = Path("/content/drive/MyDrive/naija_persona_corpus_openai")
else:
    DRIVE_BASE = Path.home() / "naija_persona_corpus_openai"
DRIVE_BASE.mkdir(parents=True, exist_ok=True)

OUTPUT_DIR    = DRIVE_BASE
RAW_DIR       = OUTPUT_DIR / "raw"
PROCESSED_DIR = OUTPUT_DIR / "processed"
METADATA_DIR  = OUTPUT_DIR / "metadata"
AUGMENTED_DIR = OUTPUT_DIR / "augmented"
CHECKPOINT_DIR = AUGMENTED_DIR / "checkpoints"
P1_CKPT_DIR    = CHECKPOINT_DIR / "pipeline1"
P2_CKPT_DIR    = CHECKPOINT_DIR / "pipeline2"
P3_CKPT_DIR    = CHECKPOINT_DIR / "pipeline3"
for d in [RAW_DIR, PROCESSED_DIR, METADATA_DIR, AUGMENTED_DIR,
          CHECKPOINT_DIR, P1_CKPT_DIR, P2_CKPT_DIR, P3_CKPT_DIR]:
    d.mkdir(parents=True, exist_ok=True)
print(f"✅ Drive base: {DRIVE_BASE}")

In [ ]:
# OPENAI API KEY
if IN_COLAB:
    try:
        from google.colab import userdata
        os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
        print("✅ OPENAI_API_KEY loaded from Colab Secrets")
    except Exception:
        pass

if not os.environ.get("OPENAI_API_KEY"):
    from getpass import getpass
    os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY (sk-…): ")
    print("✅ OPENAI_API_KEY set")

assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY required"

## 1. Settings

In [ ]:
# GENERATION SETTINGS
NUM_P1_RECORDS = 2000   # seed-grounded from Jumia reviews
NUM_P2_RECORDS = 1000   # pure synthetic
BATCH_SIZE     = 500    # save to Drive after each batch
JUDGE_SAMPLE   = 200    # records to quality-score
CONCURRENCY    = 20     # parallel OpenAI requests; bump to 50+ if you're on tier 3+

# Models — gpt-4o for both gives best Pidgin quality + consistent judge bar.
# Swap to gpt-4o-mini if you want ~5× cheaper (Pidgin quality drops slightly).
PRIMARY_MODEL = "gpt-4o"
JUDGE_MODEL   = "gpt-4o"

# Estimated cost for 3k corpus on gpt-4o
# - P1: 2000 records × 2 calls (rate + rewrite) = 4000 × ~$0.005/call = $20
# - P2: 1000 records × 1 call                  = 1000 × ~$0.005/call = $5
# - Judge: 200 × 1 call                        = 200 × ~$0.003/call = $1
# Total: ~$26

print(f"📊 Targets: P1={NUM_P1_RECORDS:,}  P2={NUM_P2_RECORDS:,}  Total={NUM_P1_RECORDS + NUM_P2_RECORDS:,}")
print(f"   batch_size={BATCH_SIZE:,}  concurrency={CONCURRENCY}")
print(f"   primary={PRIMARY_MODEL}  judge={JUDGE_MODEL}")

## 2. OpenAI Async Client + Rate-Limit Backoff

In [ ]:
# Async OpenAI client with exponential backoff on 429/500/timeout
client = AsyncOpenAI(api_key=os.environ["OPENAI_API_KEY"], timeout=60.0, max_retries=0)

async def call_openai(
    model: str,
    system: str,
    user: str,
    max_tokens: int = 500,
    temperature: float = 0.75,
    response_format=None,
    max_retries: int = 5,
) -> str:
    """One LLM call with retry on transient errors."""
    last_err = None
    for attempt in range(max_retries):
        try:
            kwargs = dict(
                model=model,
                messages=[
                    {"role": "system", "content": system},
                    {"role": "user", "content": user},
                ],
                max_tokens=max_tokens,
                temperature=temperature,
            )
            if response_format:
                kwargs["response_format"] = response_format
            resp = await client.chat.completions.create(**kwargs)
            return resp.choices[0].message.content
        except Exception as e:
            last_err = e
            msg = str(e).lower()
            # Retry on rate-limit, server error, or timeout
            if "rate" in msg or "429" in msg or "500" in msg or "503" in msg or "timeout" in msg:
                if attempt < max_retries - 1:
                    sleep_s = 4 * (2 ** attempt) + random.uniform(0, 1)
                    await asyncio.sleep(sleep_s)
                    continue
            # Non-retryable error — re-raise
            raise
    raise last_err

# Smoke test
async def _smoke():
    r = await call_openai(PRIMARY_MODEL, "You are helpful.", "Say 'OPENAI OK'.", max_tokens=10, temperature=0)
    return r.strip()
result = asyncio.run(_smoke())
print(f"✅ OpenAI reachable: {result}")

## 3. Batched Generation Helper — Auto-Save + Resume

In [ ]:
def count_existing_checkpoints(ckpt_dir: Path):
    files = sorted(ckpt_dir.glob("batch_*.parquet"))
    total = 0
    for f in files:
        try: total += len(pd.read_parquet(f))
        except: pass
    return len(files), total, files

def load_all_checkpoints(ckpt_dir: Path) -> pd.DataFrame:
    files = sorted(ckpt_dir.glob("batch_*.parquet"))
    if not files: return pd.DataFrame()
    dfs = []
    for f in files:
        try: dfs.append(pd.read_parquet(f))
        except Exception as e: print(f"  ⚠️ skipping {f.name}: {e}")
    out = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
    print(f"  📦 loaded {len(out):,} from {len(dfs)} checkpoints")
    return out

async def generate_batched(
    *, generate_one_fn, total_records: int, batch_size: int,
    ckpt_dir: Path, dataset_name: str, concurrency: int = CONCURRENCY,
):
    """
    generate_one_fn: async fn that takes an index and returns a dict row.
    Runs `concurrency` parallel calls. Saves every `batch_size` rows to Drive.
    """
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    done_batches, done_records, _ = count_existing_checkpoints(ckpt_dir)

    if done_records >= total_records:
        print(f"✅ {dataset_name}: complete ({done_records:,} on Drive)")
        return load_all_checkpoints(ckpt_dir)

    remaining = total_records - done_records
    if done_records > 0:
        print(f"🔄 {dataset_name}: resuming from batch {done_batches+1} ({done_records:,} done, {remaining:,} to go)")
    else:
        print(f"🚀 {dataset_name}: starting fresh ({total_records:,} target, concurrency={concurrency})")

    batch_num = done_batches + 1
    generated = done_records
    t_start = time.time()
    sem = asyncio.Semaphore(concurrency)

    async def _wrap(idx):
        async with sem:
            try:
                return await generate_one_fn(idx)
            except Exception as e:
                return {"_error": f"{type(e).__name__}: {str(e)[:200]}", "_idx": idx}

    while generated < total_records:
        this_n = min(batch_size, total_records - generated)
        t_batch = time.time()
        print(f"\n📦 batch {batch_num} — {this_n:,} records ({100*generated/total_records:.1f}% done)")

        tasks = [_wrap(generated + i) for i in range(this_n)]
        results = []
        async for fut in tqdm(asyncio.as_completed(tasks), total=len(tasks), desc=f"batch {batch_num}") if False else _aiter_completed(tasks):
            results.append(await fut)

        # Filter errors
        ok = [r for r in results if r and "_error" not in r]
        errs = [r for r in results if r and "_error" in r]
        if errs:
            print(f"  ⚠️ {len(errs)} errors in this batch — keeping the {len(ok)} good ones")
            for e in errs[:3]:
                print(f"      • {e.get('_error', 'unknown')[:120]}")

        if not ok:
            print("  ⚠️ entire batch failed; retrying in 30s")
            await asyncio.sleep(30)
            continue

        df = pd.DataFrame(ok)
        ckpt_pq = ckpt_dir / f"batch_{batch_num:03d}.parquet"
        ckpt_jl = ckpt_dir / f"batch_{batch_num:03d}.jsonl"
        df.to_parquet(ckpt_pq, index=False)
        with jsonlines.open(ckpt_jl, mode='w') as w:
            for _, row in df.iterrows():
                w.write(row.to_dict())

        elapsed = time.time() - t_batch
        generated += len(df)
        rate = len(df) / elapsed * 60 if elapsed else 0
        print(f"  ✅ {len(df):,} records in {elapsed:.0f}s ({rate:.0f}/min) → {ckpt_pq.name}")

        # ETA
        sess_done = generated - done_records
        if sess_done > 0:
            eta = (total_records - generated) * (time.time() - t_start) / sess_done / 60
            print(f"  ⏱️ ETA: ~{eta:.0f} min")
        batch_num += 1

    print(f"\n✅ session in {(time.time()-t_start)/60:.1f}min")
    return load_all_checkpoints(ckpt_dir)

async def _aiter_completed(coros):
    """async-iter wrapper for asyncio.as_completed."""
    for fut in asyncio.as_completed(coros):
        yield fut

print("✅ batched-generation helpers ready")

## 4. Prepare Seed Data on Drive

In [ ]:
JUMIA_REVIEWS_URL = (
    "https://raw.githubusercontent.com/"
    "aymane-maghouti/Sentiment-Analysis-for-Jumia-Reviews-and-Smartphone-Price-Prediction-System/"
    "main/Main/Sentiment_Analysis_for_Jumia_Reviews/final_data.csv"
)
SEED_REVIEWS_PATH  = RAW_DIR / "jumia_reviews_seed.parquet"
SEED_PRODUCTS_PATH = RAW_DIR / "jumia_products_seed.parquet"

# --- Real Jumia reviews ---
if SEED_REVIEWS_PATH.exists():
    df_jumia_reviews = pd.read_parquet(SEED_REVIEWS_PATH)
    print(f"✅ Jumia reviews seed: {len(df_jumia_reviews):,}")
else:
    print("📥 downloading Jumia CSV (~15MB)...")
    raw_csv = RAW_DIR / "jumia_reviews_raw.csv"
    with httpx.Client(timeout=120.0) as cli:
        r = cli.get(JUMIA_REVIEWS_URL); r.raise_for_status()
        raw_csv.write_bytes(r.content)
    rows = []
    with raw_csv.open(encoding="utf-8", errors="replace") as fh:
        rd = csv.reader(fh); next(rd, None)
        for r in rd:
            if len(r) < 2: continue
            text = " ".join(r[0].split()).strip()
            if not (50 <= len(text) <= 1500): continue
            try: tgt = int(r[1])
            except: continue
            rows.append({"review_text": text, "binary_sentiment": tgt})
    df_jumia_reviews = pd.DataFrame(rows)
    df_jumia_reviews.to_parquet(SEED_REVIEWS_PATH, index=False)
    print(f"✅ saved {len(df_jumia_reviews):,} reviews")

# --- Product catalog ---
if SEED_PRODUCTS_PATH.exists():
    df_jumia_products = pd.read_parquet(SEED_PRODUCTS_PATH)
    print(f"✅ Jumia products: {len(df_jumia_products):,}")
else:
    print("📥 downloading Idowenst product catalog from HF...")
    from datasets import load_dataset
    ds = load_dataset("Idowenst/jumia_dataset", split="train")
    df_jumia_products = pd.DataFrame([
        {"title": r.get("name",""), "category": r.get("category",""),
         "description": (r.get("description") or "")[:600], "price_naira": r.get("price")}
        for r in ds if r.get("name")
    ])
    df_jumia_products.to_parquet(SEED_PRODUCTS_PATH, index=False)
    print(f"✅ saved {len(df_jumia_products):,} products")

## 5. Pipeline Constants — Personas, Registers, Categories

In [ ]:
PERSONAS = ["chinwe_owerri", "tunde_lagos", "aisha_kano", "femi_abuja", "ifeoma_ph"]
REGISTERS = [
    "nigerian_pidgin", "nigerian_pidgin",          # weighted up
    "code_mixed", "code_mixed",                    # weighted up
    "nigerian_english",
    "standard_english",
]
REGISTERS_SYNTHETIC = [
    "nigerian_pidgin", "nigerian_pidgin",
    "code_mixed", "code_mixed", "code_mixed",      # over-sample in synthetic
    "nigerian_english",
    "standard_english",
]
CATEGORIES = [
    "phone_tablet", "electronics", "appliances", "computing",
    "health_beauty", "fashion", "home_office", "baby_products",
    "sports_outdoor", "automobile",
]
RATINGS = ["1", "2", "3", "4", "5"]

PERSONA_GUIDE = (
    "PERSONAS:\n"
    "chinwe_owerri: Owerri Gen-Z university student, Igbo+English code-mix (nna, biko, ahn ahn), communal-hedonic.\n"
    "tunde_lagos: Lagos market trader 40s, Pidgin-heavy, utilitarian, focused on value/durability.\n"
    "aisha_kano: Kano teacher, measured Nigerian English with Muslim framing (Alhamdulillah, wallahi).\n"
    "femi_abuja: Abuja banker, standard Nigerian English, low-intensity, individualist.\n"
    "ifeoma_ph: PH fashion entrepreneur + Nollywood fan, vibrant Nigerian English with film vocab."
)
REGISTER_GUIDE = (
    "REGISTERS:\n"
    "nigerian_pidgin: Use Pidgin markers naturally: 'abeg', 'no cap', 'wahala', 'e shock me', 'scatter', 'dey', 'go'.\n"
    "code_mixed: Mix Nigerian English with Yoruba/Hausa/Igbo (ahn ahn, haba, nna, biko, owambe, wahala).\n"
    "nigerian_english: Nigerian English markers ('well done sir', 'no shaking', 'sharp sharp', 'sef', 'now').\n"
    "standard_english: Clear standard English, no slang, no Pidgin."
)

print("✅ persona / register / category constants loaded")

## 6. Pipeline 1 — Seed-Grounded: Real Jumia review → (persona, rating, register, rewritten review)

In [ ]:
# Cache the Jumia reviews + products into module-scope lists for fast sampling
P1_SOURCE = df_jumia_reviews.to_dict('records')
random.seed(42); random.shuffle(P1_SOURCE)

RATING_SYS = (
    "You calibrate Nigerian product reviewers. Given a Jumia review and its binary sentiment, "
    "output ONLY a single digit 1-5 matching the text's intensity."
)
REWRITE_SYS = (
    "You rewrite real Nigerian product reviews in specific persona voices and registers. "
    "Preserve sentiment and key points. Sound like a real Nigerian customer. "
    "Output ONLY the rewritten review text — no labels, no quotes, no preamble."
)

async def p1_generate_one(idx: int) -> dict:
    src = P1_SOURCE[idx % len(P1_SOURCE)]
    persona = random.choice(PERSONAS)
    register = random.choice(REGISTERS)
    category = random.choice(CATEGORIES)

    # Step 1: refine binary sentiment → 1-5 rating
    rating_prompt = (
        f"Review: {src['review_text']}\n\n"
        f"Binary sentiment: {src['binary_sentiment']} (+1=pos, -1=neg)\n\n"
        "Scale: 1=rubbish/scam, 2=clearly negative, 3=mixed, 4=clearly positive, 5='e too much'/scatter.\n\n"
        "Output ONLY one digit (1, 2, 3, 4, or 5)."
    )
    rating_raw = await call_openai(PRIMARY_MODEL, RATING_SYS, rating_prompt, max_tokens=5, temperature=0)
    m = re.search(r'[1-5]', rating_raw or "")
    rating = int(m.group()) if m else (4 if src['binary_sentiment']==1 else 2)

    # Step 2: rewrite review in persona voice + register
    rewrite_prompt = (
        f"ORIGINAL REVIEW: {src['review_text']}\n\n"
        f"PERSONA: {persona}\n"
        f"REGISTER: {register}\n"
        f"CATEGORY: {category}\n"
        f"TARGET RATING: {rating}/5\n\n"
        "Rewrite the review:\n"
        "- Keep the SAME sentiment and key points\n"
        "- Match the persona's voice + register strictly\n"
        "- 2-5 sentences, sound real (no AI tells)\n\n"
        f"{PERSONA_GUIDE}\n\n{REGISTER_GUIDE}\n\n"
        "OUTPUT THE REVIEW ONLY:"
    )
    review = await call_openai(PRIMARY_MODEL, REWRITE_SYS, rewrite_prompt, max_tokens=400, temperature=0.75)
    return {
        "persona_id": persona,
        "register_tier": register,
        "product_category": category,
        "refined_rating": rating,
        "persona_review": (review or "").strip(),
        "source_review": src['review_text'][:300],
        "source_sentiment": src['binary_sentiment'],
    }

print("✅ Pipeline 1 generator defined")

In [ ]:
# Pipeline 1 — preview 5 samples
print("🔍 generating 5 P1 preview samples...")
preview_p1 = asyncio.run(asyncio.gather(*[p1_generate_one(i) for i in range(5)]))
for i, p in enumerate(preview_p1):
    print(f"\n=== sample {i+1} ===")
    print(f"  persona: {p['persona_id']} | register: {p['register_tier']} | rating: {p['refined_rating']}/5")
    print(f"  ORIGINAL: {p['source_review'][:150]}")
    print(f"  REWRITE:  {p['persona_review'][:200]}")

In [ ]:
# Full Pipeline 1 — batched + resumable
df_p1 = asyncio.run(generate_batched(
    generate_one_fn=p1_generate_one,
    total_records=NUM_P1_RECORDS,
    batch_size=BATCH_SIZE,
    ckpt_dir=P1_CKPT_DIR,
    dataset_name="naija_p1_seed",
    concurrency=CONCURRENCY,
))
print(f"\n✅ Pipeline 1: {len(df_p1):,} records")
if len(df_p1) > 0:
    df_p1.to_parquet(AUGMENTED_DIR / "pipeline1_seed_grounded.parquet", index=False)

## 7. Pipeline 2 — Pure Synthetic: persona × product × register × stratified rating

In [ ]:
P2_SOURCE = df_jumia_products.to_dict('records')
random.shuffle(P2_SOURCE)

SYNTH_SYS = (
    "You generate authentic Nigerian Jumia product reviews matching the user's persona voice, "
    "register tier, and target rating. Sound like a real Nigerian customer. "
    "Output ONLY the review text — no labels, no quotes, no preamble."
)

async def p2_generate_one(idx: int) -> dict:
    product = P2_SOURCE[idx % len(P2_SOURCE)]
    persona = random.choice(PERSONAS)
    register = random.choice(REGISTERS_SYNTHETIC)
    rating = random.choice(RATINGS)

    prompt = (
        f"PRODUCT (Jumia):\n"
        f"Title: {product.get('title','')}\n"
        f"Category: {product.get('category','')}\n"
        f"Description: {(product.get('description') or '')[:300]}\n"
        f"Price: ₦{product.get('price_naira','?')}\n\n"
        f"PERSONA: {persona}\n"
        f"REGISTER: {register}\n"
        f"TARGET RATING: {rating}/5\n\n"
        f"Write a {rating}/5 review (2-5 sentences) as this persona. "
        "Don't state the number — let intensity show through tone.\n\n"
        f"{PERSONA_GUIDE}\n\n{REGISTER_GUIDE}\n\n"
        "OUTPUT THE REVIEW ONLY:"
    )
    review = await call_openai(PRIMARY_MODEL, SYNTH_SYS, prompt, max_tokens=400, temperature=0.8)
    return {
        "persona_id": persona,
        "register_tier": register,
        "target_rating": int(rating),
        "title": product.get('title',''),
        "category": product.get('category',''),
        "persona_review": (review or "").strip(),
    }

print("✅ Pipeline 2 generator defined")

In [ ]:
# Pipeline 2 preview
print("🔍 generating 5 P2 preview samples...")
preview_p2 = asyncio.run(asyncio.gather(*[p2_generate_one(i) for i in range(5)]))
for i, p in enumerate(preview_p2):
    print(f"\n=== sample {i+1} ===")
    print(f"  persona: {p['persona_id']} | register: {p['register_tier']} | rating: {p['target_rating']}/5")
    print(f"  product: {p['title'][:80]}")
    print(f"  review:  {p['persona_review'][:200]}")

In [ ]:
# Full Pipeline 2 — batched + resumable
df_p2 = asyncio.run(generate_batched(
    generate_one_fn=p2_generate_one,
    total_records=NUM_P2_RECORDS,
    batch_size=BATCH_SIZE,
    ckpt_dir=P2_CKPT_DIR,
    dataset_name="naija_p2_synthetic",
    concurrency=CONCURRENCY,
))
print(f"\n✅ Pipeline 2: {len(df_p2):,} records")
if len(df_p2) > 0:
    df_p2.to_parquet(AUGMENTED_DIR / "pipeline2_synthetic.parquet", index=False)

## 8. Pipeline 3 — LLM-as-Judge Quality Scoring

In [ ]:
# Combine P1 + P2 into a unified format for judging
all_generated = []
if 'df_p1' in dir() and df_p1 is not None and len(df_p1) > 0:
    p1 = df_p1[['persona_review', 'persona_id', 'register_tier', 'product_category', 'refined_rating']].copy()
    p1.columns = ['review', 'persona_id', 'register_tier', 'product_category', 'rating']
    p1['rating'] = pd.to_numeric(p1['rating'], errors='coerce').clip(1,5).fillna(3).astype(int)
    p1['product_title'] = ''
    p1['pipeline'] = 'seed_grounded'
    all_generated.append(p1)
if 'df_p2' in dir() and df_p2 is not None and len(df_p2) > 0:
    p2 = df_p2[['persona_review', 'persona_id', 'register_tier', 'category', 'target_rating', 'title']].copy()
    p2.columns = ['review', 'persona_id', 'register_tier', 'product_category', 'rating', 'product_title']
    p2['rating'] = pd.to_numeric(p2['rating'], errors='coerce').clip(1,5).fillna(3).astype(int)
    p2['pipeline'] = 'synthetic'
    all_generated.append(p2)

df_all = pd.concat(all_generated, ignore_index=True) if all_generated else pd.DataFrame()
df_all = df_all.dropna(subset=['review'])
df_all = df_all[df_all['review'].str.len().between(50, 1500)]
df_all = df_all.drop_duplicates(subset=['review'], keep='first').reset_index(drop=True)

print(f"📊 Combined corpus: {len(df_all):,}")
print(f"   by pipeline:  {dict(df_all['pipeline'].value_counts())}")
print(f"   by register:  {dict(df_all['register_tier'].value_counts())}")
print(f"   by rating:    {dict(sorted(df_all['rating'].value_counts().items()))}")

df_all.to_parquet(AUGMENTED_DIR / "combined_pre_judge.parquet", index=False)

In [ ]:
# Judge generator — scores 4 dimensions on 0-4 scale via JSON output
JUDGE_SYS = (
    "You evaluate Nigerian product reviews on 4 dimensions: register_fidelity, persona_authenticity, "
    "rating_coherence, naturalness. Return STRICT JSON with integer scores 0-4 for each. "
    "4=perfect, 3=good, 2=partial, 1=weak, 0=bad."
)
JUDGE_PROMPT_TPL = (
    "Evaluate this Nigerian product review:\n\n"
    "PERSONA: {persona_id}\n"
    "DECLARED REGISTER: {register_tier}\n"
    "TARGET RATING: {rating}/5\n"
    "REVIEW: {review}\n\n"
    "Score 0-4 on each dimension:\n"
    "- register_fidelity: does language match the declared register tier?\n"
    "- persona_authenticity: sounds like the declared persona archetype?\n"
    "- rating_coherence: text intensity matches the target rating?\n"
    "- naturalness: reads as real Nigerian (not AI)?\n\n"
    'Return STRICT JSON: {{"register_fidelity": int, "persona_authenticity": int, '
    '"rating_coherence": int, "naturalness": int}}'
)

JUDGE_N = min(JUDGE_SAMPLE, len(df_all))
JUDGE_INPUT = df_all.sample(n=JUDGE_N, random_state=42).reset_index(drop=True).to_dict('records')

async def judge_generate_one(idx: int) -> dict:
    src = JUDGE_INPUT[idx]
    prompt = JUDGE_PROMPT_TPL.format(
        persona_id=src['persona_id'],
        register_tier=src['register_tier'],
        rating=src['rating'],
        review=src['review'][:1200],
    )
    raw = await call_openai(
        JUDGE_MODEL, JUDGE_SYS, prompt,
        max_tokens=200, temperature=0.1,
        response_format={"type": "json_object"},
    )
    try:
        scores = json.loads(raw)
    except Exception:
        scores = {}
    return {
        "review": src['review'],
        "persona_id": src['persona_id'],
        "register_tier": src['register_tier'],
        "rating": src['rating'],
        "register_fidelity":     int(scores.get('register_fidelity', 0) or 0),
        "persona_authenticity":  int(scores.get('persona_authenticity', 0) or 0),
        "rating_coherence":      int(scores.get('rating_coherence', 0) or 0),
        "naturalness":           int(scores.get('naturalness', 0) or 0),
    }

df_judged = asyncio.run(generate_batched(
    generate_one_fn=judge_generate_one,
    total_records=JUDGE_N,
    batch_size=BATCH_SIZE,
    ckpt_dir=P3_CKPT_DIR,
    dataset_name="naija_p3_judge",
    concurrency=CONCURRENCY,
))

score_cols = ['register_fidelity', 'persona_authenticity', 'rating_coherence', 'naturalness']
for c in score_cols:
    df_judged[c] = pd.to_numeric(df_judged[c], errors='coerce')
df_judged['composite_score'] = df_judged[score_cols].mean(axis=1)

print("\n📊 QUALITY SCORES")
for c in score_cols:
    print(f"   {c:25s}  mean={df_judged[c].mean():.2f}  std={df_judged[c].std():.2f}")
print(f"   {'composite':25s}  mean={df_judged['composite_score'].mean():.2f}")
hq = df_judged[df_judged['composite_score'] >= 3.0]
print(f"\n   high-quality (≥3.0): {len(hq):,}/{len(df_judged):,} ({100*len(hq)/max(len(df_judged),1):.1f}%)")
df_judged.to_parquet(AUGMENTED_DIR / "quality_scored_sample.parquet", index=False)

## 9. Final Filter + Split + Export to Drive

In [ ]:
df_final = df_all.copy()
if 'composite_score' in df_judged.columns and len(df_judged):
    score_map = df_judged.set_index('review')['composite_score'].to_dict()
    df_final['composite_score'] = df_final['review'].map(score_map)
    n_before = len(df_final)
    keep = (df_final['composite_score'].isna()) | (df_final['composite_score'] >= 3.0)
    df_final = df_final[keep].reset_index(drop=True)
    print(f"🔍 judge filter: {n_before:,} → {len(df_final):,}")

# Stratified 90/5/5 split by register
_r = random.Random(42)
buckets = defaultdict(list)
for _, row in df_final.iterrows():
    buckets[row['register_tier']].append(row.to_dict())
train, val, test = [], [], []
for tier, items in buckets.items():
    _r.shuffle(items)
    n = len(items); n_v = max(1, n//20); n_t = max(1, n//20)
    val   += items[:n_v]
    test  += items[n_v:n_v+n_t]
    train += items[n_v+n_t:]
_r.shuffle(train); _r.shuffle(val); _r.shuffle(test)
print(f"📊 splits: train={len(train):,}  val={len(val):,}  test={len(test):,}")

INSTRUCTION = (
    "Simulate the review behaviour of the following Nigerian user reviewing the described "
    "product. Generate the rating (1-5) and review text exactly as this user would write it. "
    "Match the user's register tier and cultural framing."
)

def to_alpaca(r):
    persona = json.dumps({"persona_id": r['persona_id'], "register_tier": r['register_tier']}, ensure_ascii=False)
    product = json.dumps({"title": r.get('product_title',''), "category": r.get('product_category','')}, ensure_ascii=False)
    output  = json.dumps({"rating": int(r['rating']), "review": r['review']}, ensure_ascii=False)
    return {
        "instruction": INSTRUCTION,
        "input": f"### Persona\n{persona}\n\n### Product\n{product}\n\n### Register tier\n{r['register_tier']}",
        "output": output,
    }

def to_sharegpt(r):
    persona = json.dumps({"persona_id": r['persona_id'], "register_tier": r['register_tier']}, ensure_ascii=False)
    product = json.dumps({"title": r.get('product_title',''), "category": r.get('product_category','')}, ensure_ascii=False)
    user = f"{INSTRUCTION}\n\n### Persona\n{persona}\n\n### Product\n{product}\n\n### Register tier\n{r['register_tier']}"
    asst = json.dumps({"rating": int(r['rating']), "review": r['review']}, ensure_ascii=False)
    return {"conversations": [{"from":"human","value":user}, {"from":"gpt","value":asst}]}

for name, rows in [("train", train), ("val", val), ("test", test)]:
    with jsonlines.open(PROCESSED_DIR / f"v1_{name}_alpaca.jsonl", mode='w') as w:
        for r in rows: w.write(to_alpaca(r))
    with jsonlines.open(PROCESSED_DIR / f"v1_{name}_sharegpt.jsonl", mode='w') as w:
        for r in rows: w.write(to_sharegpt(r))
    pd.DataFrame(rows).to_parquet(PROCESSED_DIR / f"v1_{name}_full.parquet", index=False)
    print(f"  💾 {name}: {len(rows):,} → alpaca + sharegpt + parquet")

# Combined full corpus (no split)
all_rows = train + val + test
with jsonlines.open(PROCESSED_DIR / "final_alpaca_format.jsonl", mode='w') as w:
    for r in all_rows: w.write(to_alpaca(r))
with jsonlines.open(PROCESSED_DIR / "final_sharegpt_format.jsonl", mode='w') as w:
    for r in all_rows: w.write(to_sharegpt(r))
pd.DataFrame(all_rows).to_parquet(PROCESSED_DIR / "final_full_dataset.parquet", index=False)

# Statistics
final_stats = {
    "total_pairs": len(all_rows),
    "splits": {"train": len(train), "val": len(val), "test": len(test)},
    "by_pipeline": dict(pd.DataFrame(all_rows)['pipeline'].value_counts()),
    "by_register": dict(pd.DataFrame(all_rows)['register_tier'].value_counts()),
    "by_rating":   dict(sorted(pd.DataFrame(all_rows)['rating'].value_counts().items())),
    "generated_at": datetime.now().isoformat(),
    "generated_with": f"OpenAI {PRIMARY_MODEL} + {JUDGE_MODEL} judge",
}
(METADATA_DIR / "final_statistics.json").write_text(json.dumps(final_stats, indent=2, default=str))

print(f"\n{'='*60}")
print("🏆 FINAL STATS")
print(f"{'='*60}")
print(json.dumps(final_stats, indent=2, default=str))

In [ ]:
# Output directory tree
print("\n📁 Output Directory:")
print("="*60)
for root, dirs, files in os.walk(OUTPUT_DIR):
    level = root.replace(str(OUTPUT_DIR), '').count(os.sep)
    indent = '  ' * level
    print(f"{indent}📂 {os.path.basename(root)}/")
    for f in sorted(files):
        fp = Path(root) / f
        size = fp.stat().st_size / (1024*1024)
        print(f"{indent}  📄 {f} ({size:.2f} MB)")

print(f"""
{'='*70}
✅ NAIJA CORPUS COMPLETE — built via OpenAI {PRIMARY_MODEL}
{'='*70}

Drive location:
   {OUTPUT_DIR}

For fine-tuning (open 02_finetune.ipynb on a GPU runtime):
   processed/v1_train_alpaca.jsonl
   processed/v1_val_alpaca.jsonl
   processed/v1_test_alpaca.jsonl

Next: switch to GPU runtime → open 02_finetune.ipynb
   (the default CORPUS_SOURCE="openai" will pick this corpus).
""")

## 🔧 Utility — Reset checkpoints (uncomment to regenerate)

In [ ]:
# import shutil
# shutil.rmtree(P1_CKPT_DIR, ignore_errors=True); P1_CKPT_DIR.mkdir(parents=True, exist_ok=True); print("🗑️ P1 wiped")
# shutil.rmtree(P2_CKPT_DIR, ignore_errors=True); P2_CKPT_DIR.mkdir(parents=True, exist_ok=True); print("🗑️ P2 wiped")
# shutil.rmtree(P3_CKPT_DIR, ignore_errors=True); P3_CKPT_DIR.mkdir(parents=True, exist_ok=True); print("🗑️ P3 wiped")